# Transacciones de Enable Banking: RAW → Bronze

## Objetivo
Cargar las transacciones de Santander y Laboral en
`openbanking.bronze.enablebanking_transactions`.

## Estrategia
- Primera ejecución: incorporar todas las páginas históricas pendientes.
- Ejecuciones posteriores: incorporar únicamente archivos pendientes.
- Reintentos: evitar duplicar observaciones ya cargadas.
- Conservar las observaciones de las ventanas solapadas.
- Registrar los archivos procesados, incluidas las páginas vacías.

## Organización
La lógica reside en `src/openbanking/bronze_transactions.py`.
Los esquemas se definen en `src/openbanking/bronze_schemas.py`.
Este notebook configura y ejecuta la carga.

## Alcance
Solo transacciones de la API. Saldos, históricos manuales,
Gmail, WhatsApp y consolidación Silver se incorporarán después.

In [0]:
# import sys
# from pathlib import Path

# current = Path.cwd()

# project_root = next(
#     (
#         path
#         for path in (current, *current.parents)
#         if (path / "src/openbanking").is_dir()
#         and (path / "databricks.yml").is_file()
#     ),
#     None,
# )

# if project_root is None:
#     raise RuntimeError("No se ha encontrado la raíz de la repo.")

# source_path = str(project_root / "src")

# if source_path not in sys.path:
#     sys.path.insert(0, source_path)

# from openbanking.bronze_transactions import ensure_tables

# ensure_tables(spark)

# print("Tablas Bronze y control de archivos preparadas.")

In [0]:
# display(spark.sql(
#     "DESCRIBE TABLE openbanking.bronze.enablebanking_transactions"
# ))

# display(spark.sql(
#     "DESCRIBE TABLE openbanking.ops.banking_ingestion_files"
# ))

In [0]:
# import importlib
# import openbanking.bronze_transactions as bronze

# importlib.reload(bronze)

# from openbanking.bronze_transactions import (
#     ensure_tables,
#     find_pending_transaction_files,
# )

# ensure_tables(spark)

# raw_path = (
#     "abfss://raw@graxianodatalake.dfs.core.windows.net/"
#     "personal/finanzas/openbanking/enablebanking/"
# )

# pending_files_df = find_pending_transaction_files(
#     spark,
#     dbutils,
#     raw_path,
# )

# display(
#     pending_files_df
#     .groupBy("bank")
#     .count()
#     .withColumnRenamed("count", "archivos_pendientes")
# )

In [0]:
# import sys
# from pathlib import Path

# current = Path.cwd()

# project_root = next(
#     (
#         path
#         for path in (current, *current.parents)
#         if (path / "src/openbanking").is_dir()
#         and (path / "databricks.yml").is_file()
#     ),
#     None,
# )

# if project_root is None:
#     raise RuntimeError(
#         "Abre y ejecuta el notebook desde notebooks/ "
#         "dentro de la carpeta Git graxiano-openbanking."
#     )

# source_path = str(project_root / "src")

# if source_path not in sys.path:
#     sys.path.insert(0, source_path)

# import importlib
# import openbanking.bronze_transactions as bronze

# importlib.reload(bronze)

# raw_path = (
#     "abfss://raw@graxianodatalake.dfs.core.windows.net/"
#     "personal/finanzas/openbanking/enablebanking/"
# )

# print("Módulo cargado:", bronze.__file__)

In [0]:
# result = bronze.ingest_transactions(spark, dbutils, raw_path)
# print(result)

In [0]:
# display(spark.sql("""
#     SELECT
#         bank,
#         COUNT(*) AS observaciones,
#         COUNT(DISTINCT observation_id) AS ids_observacion_distintos
#     FROM openbanking.bronze.enablebanking_transactions
#     GROUP BY bank
#     ORDER BY bank
# """))

## Validación inicial

- Primera carga: 51 archivos y 551 observaciones.
- Laboral: 78 observaciones y 78 IDs distintos.
- Santander: 473 observaciones y 473 IDs distintos.
- Segunda ejecución: 0 archivos y 0 observaciones procesadas.

Se verificó que no había IDs de observación duplicados y que una
reejecución sin archivos nuevos no añadió registros.

Las observaciones repetidas entre ventanas se conservan en Bronze.
La conciliación de movimientos se realizará en Silver.

In [0]:
# Localizar la raíz del proyecto y habilitar los módulos de src/.
# Se ejecuta tanto manualmente como desde el job diario.

import sys
from pathlib import Path

current = Path.cwd()

project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "src/openbanking").is_dir()
        and (path / "databricks.yml").is_file()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("No se ha encontrado la raíz del proyecto.")

source_path = str(project_root / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)

from openbanking.bronze_transactions import ingest_transactions

raw_path = (
    "abfss://raw@graxianodatalake.dfs.core.windows.net/"
    "personal/finanzas/openbanking/enablebanking/"
)

In [0]:
# Crear las tablas si no existen y cargar únicamente archivos pendientes.
# La primera ejecución incorpora el histórico disponible.
# Los reintentos no duplican observaciones ya insertadas.
# Los errores se propagan para que el job registre el fallo.

result = ingest_transactions(
    spark=spark,
    dbutils=dbutils,
    raw_path=raw_path,
)

print(
    f"Archivos procesados: {result['files_processed']} | "
    f"Observaciones procesadas: {result['observations_processed']}"
)